# Maternal Assertion Verification and Evidence Network (MAVEN)

## Maternal Health Misinformation Detection Pipeline

Ingests any body of text and outputs per-chunk "misinformation scores" using
[PubMedBERT embeddings](https://huggingface.co/NeuML/pubmedbert-base-embeddings), a biomedical sentence encoder fine-tuned on PubMed abstracts.

### Architecture Flow

```
Raw text
    │
    ▼
[1] Text Segmentation. Chunks (sentence / paragraph / sliding window)
    │
    ▼
[2] PubMedBERT Embeddings. 768-dim L2-normalized vector per chunk
    │
    ▼
[3] Marker Computation ─┬─ authority_sim    cosine sim to vetted anchor centroid
                        ├─ misinfo_sim      cosine sim to misinformation anchor centroid
                        ├─ claim_delta      authority_sim − misinfo_sim
                        └─ isolation_score  Isolation Forest anomaly vs. authoritative corpus
    │
    ▼
[4] Composite Score; weighted misinfo_score within [0, 1] per chunk → flagged boolean
    │
    ▼
Output DataFrame
```

### Scale handling

| Text size | Strategy |
|---|---|
| < 300 tokens | Sentence-level chunking |
| 300–3 000 tokens | Paragraph-level chunking |
| > 3 000 tokens | Sliding-window chunking (configurable window + stride) |
| Corpus / batch | Batched encoding via `sentence-transformers` |

In [1]:
!pip install -q sentence-transformers scikit-learn nltk pandas numpy torch

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

## 1. Text Segmentation

Splits a raw text body into semantically coherent chunks before embedding.

| Mode | Granularity | When to use |
|---|---|---|
| `sentence` | Finest | Short passages; high-precision flagging |
| `paragraph` | Default | Article-length text |
| `sliding_window` | Fixed token windows + overlap | Long documents; prevents context loss at boundaries |

`chunk_text(text, mode='auto')` dispatches automatically based on approximate token count.
The stride in sliding-window mode controls how much adjacent chunks overlap. Larger overlap preserves more cross-boundary context at the cost of redundant scoring.

In [2]:
import re
from typing import List, Tuple
from nltk.tokenize import sent_tokenize

# Thresholds for auto-dispatch 
SENTENCE_THRESHOLD  = 300    # tokens: below → sentence mode
PARAGRAPH_THRESHOLD = 3_000  # tokens: below → paragraph mode; above → sliding window


def _approx_tokens(text: str) -> int:
    return len(text.split())


def _by_sentence(text: str) -> List[str]:
    return [s.strip() for s in sent_tokenize(text) if len(s.strip()) > 20]


def _by_paragraph(text: str) -> List[str]:
    paras = [p.strip() for p in re.split(r'\n{2,}', text)]
    return [p for p in paras if len(p) > 40]


def _by_sliding_window(
    text: str,
    window: int = 200, # words per chunk
    stride: int = 100, # words between chunk starts (50 % overlap by default)
) -> List[str]:
    words = text.split()
    chunks = []
    for i in range(0, len(words), stride):
        chunk = ' '.join(words[i : i + window])
        if len(chunk) > 40:
            chunks.append(chunk)
        if i + window >= len(words):
            break
    return chunks


def chunk_text(
    text: str,
    mode: str = 'auto', # 'auto' | 'sentence' | 'paragraph' | 'sliding_window'
    window: int = 200,
    stride: int = 100,
) -> Tuple[List[str], str]:
    """Return (chunks, mode_used)."""
    if mode == 'auto':
        n = _approx_tokens(text)
        if n < SENTENCE_THRESHOLD:
            mode = 'sentence'
        elif n < PARAGRAPH_THRESHOLD:
            mode = 'paragraph'
        else:
            mode = 'sliding_window'

    dispatch = {
        'sentence':       lambda: _by_sentence(text),
        'paragraph':      lambda: _by_paragraph(text),
        'sliding_window': lambda: _by_sliding_window(text, window, stride),
    }
    return dispatch[mode](), mode


# sanity check 
_sample = (
    'Prenatal folic acid reduces neural tube defect risk.\n\n'
    'Vaccines during pregnancy are safe and recommended by the CDC and WHO.\n\n'
    'Misinformation claims that ultrasounds harm fetal brain development.'
)
for _mode in ('sentence', 'paragraph', 'sliding_window'):
    _chunks, _used = chunk_text(_sample, mode=_mode)
    print(f'{_used:>16} → {len(_chunks)} chunk(s)')

        sentence → 3 chunk(s)
       paragraph → 3 chunk(s)
  sliding_window → 1 chunk(s)


## 2. PubMedBERT Embeddings

**Model:** [`NeuML/pubmedbert-base-embeddings`](https://huggingface.co/NeuML/pubmedbert-base-embeddings)
A `sentence-transformers` model fine-tuned on PubMed biomedical text pairs.
Produces **768-dimensional, L2-normalized** vectors.

**Motivation Behind PubMedBERT over general BERT:**
General-domain encoders conflate clinical terminology with colloquial usage.
PubMedBERT captures *biomedical* semantic distance, making cosine similarity
meaningful and more explanatory for distinguishing evidence-based claims from unsupported ones.

**Key properties used here:**
- Because embeddings are L2-normalized, `cosine_similarity(a, b) = dot(a, b)` — fast and numerically stable.
- Batch encoding amortizes model overhead; scale from single sentences to millions of passages by adjusting `batch_size`.

In [3]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

print('Loading PubMedBERT embeddings model...')
embed_model = SentenceTransformer('NeuML/pubmedbert-base-embeddings', device=DEVICE)
EMBED_DIM = 768
print('Model loaded.')


def embed(texts: List[str], batch_size: int = 32, show_progress: bool = False) -> np.ndarray:
    """
    Encode texts into L2-normalized 768-dim PubMedBERT embeddings.
    Returns ndarray of shape (len(texts), 768).
    Handles any corpus size via batching.
    """
    if not texts:
        return np.empty((0, EMBED_DIM))
    return embed_model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,   # L2-normalize → cosine sim = dot product
        show_progress_bar=show_progress,
        convert_to_numpy=True,
    )


# Smoke test 
_test = embed(['Prenatal care reduces maternal mortality.'])
print(f'Embedding shape: {_test.shape}  |  L2-norm: {np.linalg.norm(_test[0]):.4f}')

c:\Users\Thomas\Desktop\projects\hmd-dept-of-mch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
Loading PubMedBERT embeddings model...



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7610.78it/s]

Model loaded.
Embedding shape: (1, 768)  |  L2-norm: 1.0000


## 3. Quantifiable Misinformation Markers

Four continuous markers are computed per chunk from its PubMedBERT embedding.

| Marker | Range | High value means |
|---|---|---|
| `authority_sim` | [−1, 1] | Chunk is semantically close to verified, authoritative claims |
| `misinfo_sim` | [−1, 1] | Chunk is semantically close to known misinformation claims |
| `claim_delta` | [−2, 2] | `authority_sim − misinfo_sim`; positive -> leans authoritative, negative -> leans misinformation |
| `isolation_score` | [0, 1] | Isolation Forest anomaly score vs. authoritative corpus; high -> anomalous relative to vetted content |

### Anchor sets are now loaded from domain-grounded JSON

Anchor sets are extracted from two domain assets in `perinatal_knowledge_base/dept_mch_feed/` by `scripts/build_anchors.py`:

- **`authority_anchors.json`** (~119 anchors) — declarative recommendation sentences from the perinatal comprehensive reference (PART I authority bodies + DOMAIN 1 clinical care), filtered to those naming a guideline body (ACOG, CDC, WHO, AAP, SMFM, USPSTF, AHRQ, MedlinePlus, FDA, etc.), plus the EVIDENCE lines paired with documented misinfo claims.
- **`misinfo_anchors.json`** (~74 CLAIM/EVIDENCE/domain triples) — every documented CLAIM line in the perinatal misinformation supplement, paired with its evidence-based correction and tagged with the originating section (antenatal / intrapartum / fourth_trimester / family_planning / clinical_provider / social_*).
- **`misinfo_type_anchors.json`** — 8 hand-curated seed buckets (5–6 anchors each) for the misinfo taxonomy from John & Gorman, *J Gen Intern Med* 2024 (`unattributed_risk`, `not_aligned_with_guidelines`, `promotes_alternative_medicine`, `exaggerated_risk`, `discourages_evidence_based`, `undermines_medical_trust`, `inaccurate_biological_mechanism`, `other`).
- **`comprehensive_paragraphs.json`** (~281 paragraphs) — the full authoritative perinatal reference, used as the IsolationForest training corpus so anomaly detection learns the distribution of *real* clinical text, not just a handful of anchor sentences.

The Isolation Forest is now fit on hundreds of authoritative paragraph embeddings rather than the original 10 anchors, which materially improves its ability to flag *narrative* misinfo (persuasive framings with no falsifiable claim) — the design insight from the John & Gorman narrative review.

In [4]:
import json
import urllib.request
from pathlib import Path
from sklearn.ensemble import IsolationForest

# Anchor source: 4 JSON files emitted by scripts/build_anchors.py
# Local first; Colab fallback fetches from GitHub raw on the active branch.
ANCHOR_FILES = [
    'authority_anchors.json',
    'misinfo_anchors.json',
    'misinfo_type_anchors.json',
    'comprehensive_paragraphs.json',
]
LOCAL_ANCHOR_DIR = Path('maven_app/anchors')
GITHUB_RAW = 'https://raw.githubusercontent.com/ai-unc/health-misinformation-detection-system/main/maven_app/anchors'


def _load_json(name: str):
    """Try local first; fall back to GitHub raw (Colab-friendly)."""
    local = LOCAL_ANCHOR_DIR / name
    if local.exists():
        return json.loads(local.read_text(encoding='utf-8'))
    print(f'[anchors] {name}: not found locally, fetching from GitHub raw...')
    with urllib.request.urlopen(f'{GITHUB_RAW}/{name}') as r:
        return json.loads(r.read().decode('utf-8'))


print('Loading anchors...')
authority_anchors  = _load_json('authority_anchors.json')           # list[str]
misinfo_anchors    = _load_json('misinfo_anchors.json')             # list[{claim,evidence,domain}]
misinfo_type_seeds = _load_json('misinfo_type_anchors.json')        # dict[type_id -> list[str]]
comp_paragraphs    = _load_json('comprehensive_paragraphs.json')    # list[str] (IsoForest corpus)

print(f'  authority anchors:                {len(authority_anchors)}')
print(f'  misinfo CLAIM/EVIDENCE pairs:     {len(misinfo_anchors)}')
print(f'  misinfo type categories:          {len(misinfo_type_seeds)}')
print(f'  comprehensive paragraphs (IsoForest corpus): {len(comp_paragraphs)}')


# Embed anchors and build centroids
def _l2(v: np.ndarray) -> np.ndarray:
    return v / (np.linalg.norm(v) + 1e-10)


print('\nEmbedding authority anchors...')
authority_embs = embed(authority_anchors)
authority_centroid = _l2(authority_embs.mean(axis=0))[np.newaxis, :]   # (1, 768)

print('Embedding misinfo CLAIM lines...')
misinfo_claim_embs = embed([m['claim'] for m in misinfo_anchors])      # (n_claims, 768)
misinfo_centroid   = _l2(misinfo_claim_embs.mean(axis=0))[np.newaxis, :]

print('Embedding misinfo-type seed sets...')
misinfo_type_ids: List[str] = list(misinfo_type_seeds.keys())
misinfo_type_centroids = np.zeros((len(misinfo_type_ids), EMBED_DIM), dtype=np.float32)
for i, tid in enumerate(misinfo_type_ids):
    seed_embs = embed(misinfo_type_seeds[tid])
    misinfo_type_centroids[i] = _l2(seed_embs.mean(axis=0))

# Isolation Forest: trained on the FULL comprehensive perinatal reference,
# not on the small anchor set. This lets the anomaly detector learn the
# distribution of authoritative clinical text and flag narrative-style
# misinfo (persuasive framing without a falsifiable claim) — the John &
# Gorman (JGIM 2024) design insight.
print(f'Embedding {len(comp_paragraphs)} comprehensive paragraphs (IsoForest training corpus)...')
comp_embs = embed(comp_paragraphs)
print(f'Fitting IsolationForest on {len(comp_embs)} authoritative-text embeddings...')
iso_forest = IsolationForest(n_estimators=200, contamination='auto', random_state=42)
iso_forest.fit(comp_embs)

# Calibrate iso_score against the training-corpus distribution. Per-batch
# min/max would force a single-chunk request to iso_score=0.0, which would
# cap the composite score at 0.70 * delta_score and prevent any
# single-sentence input from ever crossing FLAG_THRESHOLD.
iso_raw_train = -iso_forest.decision_function(comp_embs)
ISO_LO, ISO_HI = np.percentile(iso_raw_train, [1, 99])
print(f'IsoForest calibration (1st/99th pctile of training raw scores): lo={ISO_LO:.4f}  hi={ISO_HI:.4f}')
print('Anchors embedded. Isolation Forest fitted.')


def compute_markers(chunk_embs: np.ndarray) -> dict:
    """
    Compute scoring signals for N chunk embeddings.

    Returns the four scalar markers used by the composite score, plus two
    per-anchor similarity matrices used to enrich flagged chunks with the
    closest documented CLAIM (matched_claim / evidence_correction) and the
    JGIM misinfo-type label (misinfo_type / misinfo_type_confidence).
    """
    auth_sim    = (chunk_embs @ authority_centroid.T).flatten()  # (N,)
    misinfo_sim = (chunk_embs @ misinfo_centroid.T).flatten()    # (N,)
    claim_delta = auth_sim - misinfo_sim                         # (N,) in [-2, 2]

    # Normalize against the cached training-corpus distribution (NOT per-batch
    # min/max — see comment above ISO_LO/ISO_HI). Clipping keeps the value
    # in [0, 1] regardless of batch composition or batch size.
    iso_raw   = -iso_forest.decision_function(chunk_embs)        # (N,)
    iso_score = np.clip((iso_raw - ISO_LO) / (ISO_HI - ISO_LO + 1e-10), 0.0, 1.0)

    per_claim_sims = chunk_embs @ misinfo_claim_embs.T           # (N, n_claims)
    per_type_sims  = chunk_embs @ misinfo_type_centroids.T       # (N, n_types)

    return {
        'authority_sim':   auth_sim,
        'misinfo_sim':     misinfo_sim,
        'claim_delta':     claim_delta,
        'isolation_score': iso_score,
        'per_claim_sims':  per_claim_sims,
        'per_type_sims':   per_type_sims,
    }

Loading anchors...
  authority anchors:                119
  misinfo CLAIM/EVIDENCE pairs:     74
  misinfo type categories:          8
  comprehensive paragraphs (IsoForest corpus): 281

Embedding authority anchors...


Embedding misinfo CLAIM lines...


Embedding misinfo-type seed sets...


Embedding 281 comprehensive paragraphs (IsoForest training corpus)...


Fitting IsolationForest on 281 authoritative-text embeddings...


IsoForest calibration (1st/99th pctile of training raw scores): lo=-0.0958  hi=-0.0299
Anchors embedded. Isolation Forest fitted.


## 4. Composite Scoring & Output

The four markers are combined into a single **`misinfo_score in [0, 1]`** per chunk.

### Formula

```
claim_delta_score = (1 − clip(claim_delta, −1, 1)) / 2     # maps [−1, 1] → [1, 0]
misinfo_score     = 0.70 × claim_delta_score + 0.30 × isolation_score
```

**Weight rationale:**
- `claim_delta` (weight 0.70) is the primary signal. It directly contrasts semantic proximity
  to authoritative vs. misinformation content.
- `isolation_score` (weight 0.30) supplements by detecting claims semantically distant from
  *any* authoritative content, catching novel misinformation outside the anchor vocabulary.

Raw `authority_sim` and `misinfo_sim` are preserved in the output DataFrame for transparency
and downstream analysis but are not included in the composite to avoid double-counting.

**Threshold:** A chunk is flagged when `misinfo_score ≥ FLAG_THRESHOLD` (default 0.60).
Threshold tuning against labeled data is deferred to Milestone 4.

### Per-flagged-chunk explainability fields

For every chunk where `flagged == True`, four additional output columns are populated (and left `None` otherwise to keep the response payload small):

| Field | What it carries |
|---|---|
| `matched_claim` | The closest documented misinfo CLAIM (cosine-argmax over the 74 CLAIM embeddings) |
| `evidence_correction` | The evidence-based correction paired with `matched_claim` in the source document |
| `misinfo_type` | The JGIM (John & Gorman 2024) taxonomy category whose seed centroid is closest to the chunk |
| `misinfo_type_confidence` | The cosine similarity of that argmax — frontends can hide low-confidence labels |

These fields turn the model from a binary flag into an *actionable* output: each flagged chunk ships with a paired evidence-based reference and a category label, so a user (or downstream UI) can immediately see *why* it was flagged and *what the evidence says instead*.

In [5]:
import pandas as pd
from typing import Optional

FLAG_THRESHOLD = 0.60  # threshold tuning against labeled data is deferred to Milestone 4


def score_text(
    text: str,
    chunk_mode: str = 'auto',
    window: int = 200,
    stride: int = 100,
    batch_size: int = 32,
    flag_threshold: float = FLAG_THRESHOLD,
) -> pd.DataFrame:
    """
    Intake any body of text; return a DataFrame of per-chunk misinformation scores.

    Columns:
        chunk                     raw text segment
        chunk_mode                segmentation strategy applied
        authority_sim             cosine sim to authoritative anchor centroid  (higher = better)
        misinfo_sim               cosine sim to misinformation anchor centroid (higher = worse)
        claim_delta               authority_sim − misinfo_sim                  (higher = better)
        isolation_score           anomaly score vs. authoritative corpus       (higher = worse)
        misinfo_score             composite misinformation score in [0, 1]     (higher = worse)
        flagged                   True when misinfo_score >= flag_threshold
        matched_claim             closest documented misinfo CLAIM (None when not flagged)
        evidence_correction       evidence-based correction paired with matched_claim (None when not flagged)
        misinfo_type              JGIM (J&G 2024) taxonomy category (None when not flagged)
        misinfo_type_confidence   cosine sim for the misinfo_type argmax (None when not flagged)
    """
    chunks, mode_used = chunk_text(text, mode=chunk_mode, window=window, stride=stride)
    if not chunks:
        return pd.DataFrame()

    chunk_embs = embed(chunks, batch_size=batch_size)
    markers    = compute_markers(chunk_embs)

    # Normalize claim_delta from [-1, 1] (clipped) → [0, 1]
    delta_clipped = np.clip(markers['claim_delta'], -1.0, 1.0)
    delta_score   = (1.0 - delta_clipped) / 2.0
    misinfo_score = 0.70 * delta_score + 0.30 * markers['isolation_score']
    flagged       = misinfo_score >= flag_threshold

    # Per-flagged-row enrichment (None elsewhere to keep payload small)
    n = len(chunks)
    matched_claim:       List[Optional[str]]   = [None] * n
    evidence_correction: List[Optional[str]]   = [None] * n
    misinfo_type:        List[Optional[str]]   = [None] * n
    misinfo_type_conf:   List[Optional[float]] = [None] * n

    if flagged.any():
        per_claim = markers['per_claim_sims']
        per_type  = markers['per_type_sims']
        for i in np.where(flagged)[0]:
            best_claim_idx = int(per_claim[i].argmax())
            matched_claim[i]       = misinfo_anchors[best_claim_idx]['claim']
            evidence_correction[i] = misinfo_anchors[best_claim_idx]['evidence']

            best_type_idx = int(per_type[i].argmax())
            misinfo_type[i]      = misinfo_type_ids[best_type_idx]
            misinfo_type_conf[i] = round(float(per_type[i][best_type_idx]), 4)

    return pd.DataFrame({
        'chunk':                   chunks,
        'chunk_mode':              mode_used,
        'authority_sim':           markers['authority_sim'].round(4),
        'misinfo_sim':             markers['misinfo_sim'].round(4),
        'claim_delta':             markers['claim_delta'].round(4),
        'isolation_score':         markers['isolation_score'].round(4),
        'misinfo_score':           misinfo_score.round(4),
        'flagged':                 flagged,
        'matched_claim':           matched_claim,
        'evidence_correction':     evidence_correction,
        'misinfo_type':            misinfo_type,
        'misinfo_type_confidence': misinfo_type_conf,
    })


# Demo
DEMO_TEXT = """
Adequate prenatal care is associated with significantly reduced rates of maternal
and infant mortality across all demographic groups.

Folic acid supplementation before and during early pregnancy prevents the majority
of neural tube defects, according to CDC guidelines.

Epidurals damage the baby's brain and always lead to a c-section. Pitocin is a
dangerous drug that causes lasting harm to the infant.

Preeclampsia is managed through blood pressure monitoring and, when necessary,
early delivery. It affects 5-8% of pregnancies worldwide.

Doctors push induction for billing convenience, not for patient safety. Trust
your body — it knows what to do.
"""

df_scores = score_text(DEMO_TEXT)

print(f'Chunks scored : {len(df_scores)}')
print(f'Flagged       : {df_scores["flagged"].sum()}')
print(f'Mean score    : {df_scores["misinfo_score"].mean():.3f}')
display(df_scores[['chunk', 'misinfo_score', 'flagged', 'misinfo_type']])

# Show one flagged chunk with its full evidence-based correction
flagged_rows = df_scores[df_scores['flagged']]
if not flagged_rows.empty:
    print('\n=== EXAMPLE FLAGGED CHUNK + EVIDENCE-BASED CORRECTION ===')
    r = flagged_rows.iloc[0]
    print(f'\nFlagged chunk:\n  "{r["chunk"]}"\n')
    print(f'Misinfo type:           {r["misinfo_type"]} (confidence {r["misinfo_type_confidence"]})')
    print(f'Closest known CLAIM:    {r["matched_claim"]}')
    print(f'\nEvidence-based correction:\n  {r["evidence_correction"]}')

Chunks scored : 8
Flagged       : 4
Mean score    : 0.601


,chunk,misinfo_score,flagged,misinfo_type
0,Adequate prenatal care is associated with sign...,0.5835,False,NaN
1,Folic acid supplementation before and during e...,0.5314,False,NaN
2,Epidurals damage the baby's brain and always l...,0.6554,True,exaggerated_risk
3,Pitocin is a\ndangerous drug that causes lasti...,0.6429,True,exaggerated_risk
4,Preeclampsia is managed through blood pressure...,0.6211,True,promotes_alternative_medicine
5,It affects 5-8% of pregnancies worldwide.,0.5535,False,NaN
6,Doctors push induction for billing convenience...,0.5637,False,NaN
7,Trust\nyour body — it knows what to do.,0.6596,True,other



=== EXAMPLE FLAGGED CHUNK + EVIDENCE-BASED CORRECTION ===

Flagged chunk:
  "Epidurals damage the baby's brain and always lead to a c-section."

Misinfo type:           exaggerated_risk (confidence 0.7187)
Closest known CLAIM:    Epidurals harm the baby through drug exposure.

Evidence-based correction:
  Not supported by evidence. Epidural medications are delivered into the epidural space with minimal systemic absorption. Studies do not show clinically significant adverse neonatal outcomes attributable to epidural analgesia. ACOG affirms the patient's right to effective pain relief in labour without a requirement for justification.
